In [2]:
# -*- coding: utf-8 -*-
import warnings, os, gc, time, json, joblib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    precision_recall_curve, average_precision_score, confusion_matrix,
    classification_report
)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from imblearn.under_sampling import RandomUnderSampler

# Intentar cargar LightGBM / XGBoost
LGBM_AVAILABLE = True
XGB_AVAILABLE  = True
try:
    from lightgbm import LGBMClassifier
except Exception:
    LGBM_AVAILABLE = False
try:
    from xgboost import XGBClassifier
except Exception:
    XGB_AVAILABLE = False


In [ ]:

# ==========================
# CONFIG
# ==========================
DATA_PATH = "../data/processed/df_ready_model.csv"
SAVE_DIR  = "../models"
os.makedirs(SAVE_DIR, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

MIN_PRECISION = 0.10   # objetivo de precisión mínima para escoger umbral
VAL_YEAR = 2023        # validación: año 2023
VAL_MONTH = 12         # validación: diciembre 2023
TEST_YEAR = 2024       # test final: año 2024

# ==========================
# Helpers
# ==========================
def headline(txt):
    print("\n" + txt)
    print("-" * len(txt))

def pr_auc(y_true, y_proba):
    try:
        return average_precision_score(y_true, y_proba)
    except Exception:
        return np.nan

def evaluate_at_threshold(y_true, y_proba, thr=0.5):
    y_pred = (y_proba >= thr).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    precision = 0.0 if (tp + fp) == 0 else tp / (tp + fp)
    recall    = 0.0 if (tp + fn) == 0 else tp / (tp + fn)
    f1 = 0.0 if (precision + recall) == 0 else 2 * precision * recall / (precision + recall)
    return {
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "precision": precision, "recall": recall, "f1": f1
    }

def pick_threshold_for_min_precision(y_true, y_proba, min_precision=0.10):
    """
    Elige el umbral más bajo que cumple la precisión mínima y,
    entre los que cumplen, maximiza el recall.
    """
    prec, rec, thr = precision_recall_curve(y_true, y_proba)
    prec, rec = prec[:-1], rec[:-1]  # alinear con thr
    ok = np.where(prec >= min_precision)[0]
    if len(ok) == 0:
        return 0.5, 0.0, 0.0
    best_i = ok[np.argmax(rec[ok])]
    return float(thr[best_i]), float(prec[best_i]), float(rec[best_i])

# ==========================
# Feature Engineering (sin fuga)
# ==========================
def add_cluster_aggregates(df_train, df_apply):
    """
    Crea y añade (usando SOLO TRAIN):
      - cluster_accident_rate (media accidente por cluster)
      - cluster_avg_temp, cluster_avg_precip (medias por cluster)
      - cluster_hour_accident_rate (media accidente por cluster+hour)
    """
    COL_TEMP = "temperature_2m (°C)"
    COL_PREC = "precipitation (mm)"

    agg_cluster = (
        df_train.groupby("cluster_id")
        .agg(
            cluster_accident_rate=("accident", "mean"),
            cluster_avg_temp=(COL_TEMP, "mean"),
            cluster_avg_precip=(COL_PREC, "mean"),
        )
        .reset_index()
    )

    agg_cluster_hour = (
        df_train.groupby(["cluster_id", "hour"])
        .agg(cluster_hour_accident_rate=("accident", "mean"))
        .reset_index()
    )

    out = df_apply.merge(agg_cluster, on="cluster_id", how="left")
    out = out.merge(agg_cluster_hour, on=["cluster_id", "hour"], how="left")

    # Rellenos con medias globales
    global_acc_rate   = float(df_train["accident"].mean())
    global_temp_mean  = float(df_train[COL_TEMP].mean())
    global_prec_mean  = float(df_train[COL_PREC].mean())
    global_hour_acc   = df_train.groupby("hour")["accident"].mean()

    out["cluster_accident_rate"]   = out["cluster_accident_rate"].fillna(global_acc_rate)
    out["cluster_avg_temp"]        = out["cluster_avg_temp"].fillna(global_temp_mean)
    out["cluster_avg_precip"]      = out["cluster_avg_precip"].fillna(global_prec_mean)

    def _fill_hour(r):
        v = r["cluster_hour_accident_rate"]
        if pd.isna(v):
            return float(global_hour_acc.get(r["hour"], global_acc_rate))
        return v
    out["cluster_hour_accident_rate"] = out.apply(_fill_hour, axis=1)
    return out

def add_cyclic_features(df):
    """
    Añade codificación cíclica a partir de columnas discretas existentes:
    - hour (24), day_of_week (7), month (12), day (31 aprox.)
    """
    two_pi = 2 * np.pi

    # asegurar tipos enteros
    for c in ["hour", "day_of_week", "month", "day"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

    if "hour" in df.columns:
        df["hour_sin"] = np.sin(two_pi * df["hour"] / 24.0)
        df["hour_cos"] = np.cos(two_pi * df["hour"] / 24.0)
    else:
        df["hour_sin"] = 0.0
        df["hour_cos"] = 0.0

    if "day_of_week" in df.columns:
        df["dow_sin"] = np.sin(two_pi * df["day_of_week"] / 7.0)
        df["dow_cos"] = np.cos(two_pi * df["day_of_week"] / 7.0)
    else:
        df["dow_sin"] = 0.0
        df["dow_cos"] = 0.0

    if "month" in df.columns:
        df["month_sin"] = np.sin(two_pi * df["month"] / 12.0)
        df["month_cos"] = np.cos(two_pi * df["month"] / 12.0)
    else:
        df["month_sin"] = 0.0
        df["month_cos"] = 0.0

    if "day" in df.columns:
        # 31 como aproximación (la periodicidad exacta por mes no es necesaria aquí)
        df["day_sin"] = np.sin(two_pi * df["day"] / 31.0)
        df["day_cos"] = np.cos(two_pi * df["day"] / 31.0)
    else:
        df["day_sin"] = 0.0
        df["day_cos"] = 0.0

    return df

# ==========================
# Carga y splits
# ==========================
headline("🚀 Iniciando pipeline de predicción de accidentes")
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Dataset cargado: {df.shape[0]:,} registros, {df.shape[1]} columnas")

print("\nDistribución de clases:")
print(f"Clase 0 (No accidente): {(df['accident']==0).sum():,} ({(df['accident']==0).mean()*100:.2f}%)")
print(f"Clase 1 (Accidente): {(df['accident']==1).sum():,} ({df['accident'].mean()*100:.2f}%)")

# Train: <=2022 | Valid: dic-2023 | Test: 2024
train_mask = df["year"] <= 2022
val_mask   = (df["year"] == VAL_YEAR) & (df["month"] == VAL_MONTH)
test_mask  = (df["year"] == TEST_YEAR)

df_train = df.loc[train_mask].copy()
df_val   = df.loc[val_mask].copy()
df_test  = df.loc[test_mask].copy()

print("\n📊 División temporal:")
print(f"Entrenamiento: {len(df_train):,} registros ({len(df_train)/len(df)*100:.1f}%)")
print(f"Validación (dic-{VAL_YEAR}): {len(df_val):,} registros")
print(f"Test ({TEST_YEAR}): {len(df_test):,} registros")
print(f"Accidentes en train: {df_train['accident'].sum():,} ({df_train['accident'].mean()*100:.2f}%)")
print(f"Accidentes en val  : {df_val['accident'].sum():,} ({df_val['accident'].mean()*100:.2f}%)")
print(f"Accidentes en test : {df_test['accident'].sum():,} ({df_test['accident'].mean()*100:.2f}%)")

# Features base + agregadas + cíclicas
BASE_FEATURES = [
    "cluster_id", "temperature_2m (°C)", "precipitation (mm)",
    "wind_speed_10m (km/h)", "Fiesta", "dia_festivo",
    "year", "month", "day", "hour", "day_of_week", "is_weekend",
]
CLUSTER_FEATS = [
    "cluster_accident_rate", "cluster_avg_temp", "cluster_avg_precip",
    "cluster_hour_accident_rate"
]
CYCLIC_FEATS = ["hour_sin","hour_cos","dow_sin","dow_cos","month_sin","month_cos","day_sin","day_cos"]
FEATURES = BASE_FEATURES + CLUSTER_FEATS + CYCLIC_FEATS

headline("🧪 Feature Engineering por cluster (evitando leakage)")
df_train_fe = add_cluster_aggregates(df_train, df_train)
df_val_fe   = add_cluster_aggregates(df_train, df_val)
df_test_fe  = add_cluster_aggregates(df_train, df_test)

# añadir cíclicas (derivadas de tus discretas)
df_train_fe = add_cyclic_features(df_train_fe)
df_val_fe   = add_cyclic_features(df_val_fe)
df_test_fe  = add_cyclic_features(df_test_fe)

X_train_full = df_train_fe[FEATURES].copy()
y_train_full = df_train_fe["accident"].astype(int).copy()

X_val  = df_val_fe[FEATURES].copy()
y_val  = df_val_fe["accident"].astype(int).copy()

X_test = df_test_fe[FEATURES].copy()
y_test = df_test_fe["accident"].astype(int).copy()

del df_train_fe, df_val_fe, df_test_fe
gc.collect()

# ==========================
# Balanceo (undersampling)
# ==========================
headline("⚖️ Creando datasets balanceados (ligeros y estables)")
balanced_sets = {}

# 1:20
try:
    rus = RandomUnderSampler(random_state=RANDOM_STATE, sampling_strategy=0.05)
    Xc, yc = rus.fit_resample(X_train_full, y_train_full)
    balanced_sets["undersampling_conservative"] = (Xc, yc)
    print(f"   ✅ USC 1:20  → {len(Xc):,} filas | pos={yc.sum():,} ({yc.mean()*100:.2f}%)")
except Exception as e:
    print(f"   ❌ UnderSampling 1:20: {e}")

# 1:10
try:
    rum = RandomUnderSampler(random_state=RANDOM_STATE, sampling_strategy=0.10)
    Xm, ym = rum.fit_resample(X_train_full, y_train_full)
    balanced_sets["undersampling_moderate"] = (Xm, ym)
    print(f"   ✅ USM 1:10  → {len(Xm):,} filas | pos={ym.sum():,} ({ym.mean()*100:.2f}%)")
except Exception as e:
    print(f"   ❌ UnderSampling 1:10: {e}")

# ==========================
# Modelos
# ==========================
def get_models():
    models = {}
    models["RandomForest_HighRecall"] = RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_split=3,
        min_samples_leaf=1,
        class_weight={0:1, 1:50},
        random_state=RANDOM_STATE, n_jobs=-1
    )
    if LGBM_AVAILABLE:
        models["LightGBM_Balanced"] = LGBMClassifier(
            n_estimators=300, learning_rate=0.05,
            num_leaves=63, min_child_samples=60,
            subsample=0.8, colsample_bytree=0.8,
            class_weight="balanced",
            random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
        )
    if XGB_AVAILABLE:
        models["XGBoost_Weighted"] = XGBClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=8,
            subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=1.0,
            scale_pos_weight=50, eval_metric="logloss",
            random_state=RANDOM_STATE, n_jobs=-1, verbosity=0
        )
    keep = [k for k in ["RandomForest_HighRecall","LightGBM_Balanced","XGBoost_Weighted"] if k in models]
    print("   Modelos:", keep)
    return {k: models[k] for k in keep}

models = get_models()

# ==========================
# Evaluación base (umbral 0.5)
# ==========================
headline("🔬 Evaluando combinaciones…")
results = []
trained = {}

for bal_name, (Xb, yb) in balanced_sets.items():
    print(f"\n  📊 Técnica: {bal_name.upper():<27} | datos={len(Xb):,} ({yb.mean()*100:.2f}% pos)")
    for mname, model in models.items():
        t0 = time.time()
        model.fit(Xb, yb)
        train_time = time.time() - t0

        # proba en TEST 2024
        proba_test = model.predict_proba(X_test)[:, 1]
        roc = roc_auc_score(y_test, proba_test)
        pr  = pr_auc(y_test, proba_test)

        met = evaluate_at_threshold(y_test, proba_test, thr=0.5)
        row = {
            "model": mname, "balance": bal_name, "thr": 0.5,
            "precision": met["precision"], "recall": met["recall"], "f1": met["f1"],
            "roc_auc": roc, "pr_auc": pr,
            "tn": met["tn"], "fp": met["fp"], "fn": met["fn"], "tp": met["tp"],
            "train_time_s": round(train_time, 2)
        }
        results.append(row)
        trained[f"{mname}__{bal_name}"] = model
        print(f"    → OK  R={met['recall']:.3f}  P={met['precision']:.3f}  F1={met['f1']:.3f}  PR_AUC={pr:.3f}")

df_res = pd.DataFrame(results).sort_values(
    ["pr_auc","recall","precision"], ascending=[False, False, False]
).reset_index(drop=True)

headline("📈 TOP resultados (orden: PR_AUC, Recall, Precision)")
print(df_res[["model","balance","thr","precision","recall","f1","roc_auc","pr_auc","tn","fp","fn","tp","train_time_s"]].head(10))

best = df_res.iloc[0].to_dict()
print("\n🏆 Mejor combo")
print(pd.Series(best))

# ==========================
# Optimización SOLO del mejor modelo
# ==========================
headline("🔧 Optimización del mejor modelo (grid/random) + guardado")

best_name   = best["model"]
best_bal    = best["balance"]
X_opt, y_opt = balanced_sets[best_bal]

if best_name == "LightGBM_Balanced" and LGBM_AVAILABLE:
    base = LGBMClassifier(
        class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
    )
    param_grid = {
        "n_estimators": [400, 700, 1000],
        "learning_rate": [0.03, 0.05, 0.1],
        "num_leaves": [63, 127, 255],
        "min_child_samples": [20, 60, 120],
        "colsample_bytree": [0.7, 0.9],
        "subsample": [0.7, 0.9]
    }
    gs = GridSearchCV(
        base, param_grid, scoring="average_precision",
        cv=3, n_jobs=-1, verbose=1
    )
    gs.fit(X_opt, y_opt)
    best_est = gs.best_estimator_
    print("\nMejores parámetros LGBM:", gs.best_params_)

elif best_name == "XGBoost_Weighted" and XGB_AVAILABLE:
    base = XGBClassifier(
        eval_metric="logloss",
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0
    )
    param_dist = {
        "n_estimators": [400, 700, 1000],
        "learning_rate": [0.03, 0.05, 0.1],
        "max_depth": [6, 8, 10],
        "subsample": [0.7, 0.9],
        "colsample_bytree": [0.7, 0.9],
        "reg_alpha": [0.0, 0.1, 0.3],
        "reg_lambda": [1.0, 2.0],
        "scale_pos_weight": [30, 40, 50, 60]
    }
    rs = RandomizedSearchCV(
        base, param_distributions=param_dist, n_iter=30,
        scoring="average_precision", cv=3, n_jobs=-1, verbose=1,
        random_state=RANDOM_STATE
    )
    rs.fit(X_opt, y_opt)
    best_est = rs.best_estimator_
    print("\nMejores parámetros XGB:", rs.best_params_)
else:
    base = RandomForestClassifier(
        class_weight={0:1,1:50}, random_state=RANDOM_STATE, n_jobs=-1
    )
    param_grid = {
        "n_estimators": [300, 500],
        "max_depth": [12, 16, 20],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2]
    }
    gs = GridSearchCV(
        base, param_grid, scoring="average_precision",
        cv=3, n_jobs=-1, verbose=1
    )
    gs.fit(X_opt, y_opt)
    best_est = gs.best_estimator_
    print("\nMejores parámetros RF:", gs.best_params_)

# Reentrenar con TODO el dataset balanceado ganador
t0 = time.time()
best_est.fit(X_opt, y_opt)
print(f"\nReentrenado mejor modelo sobre {len(X_opt):,} filas en {(time.time()-t0)/60:.2f} min")

# ==========================
# Selección de umbral con VALIDACIÓN (dic-2023) y TEST 2024
# ==========================
proba_val  = best_est.predict_proba(X_val)[:, 1]
thr_sel, p_val, r_val = pick_threshold_for_min_precision(y_val, proba_val, min_precision=MIN_PRECISION)
print(f"\nUmbral elegido en VALIDACIÓN (prec≥{MIN_PRECISION:.2f}): thr={thr_sel:.4f}  | prec_val={p_val:.3f}  rec_val={r_val:.3f}")

# Métricas de referencia con probabilidades (sin umbral)
proba_test = best_est.predict_proba(X_test)[:, 1]
print(f"\n== Métricas de referencia (probabilidades sin umbral) ==")
print(f"Test 2024: PR AUC = {average_precision_score(y_test, proba_test):.6f} | ROC AUC = {roc_auc_score(y_test, proba_test):.6f}")

# Evaluación en TEST 2024 con el umbral elegido
ev = evaluate_at_threshold(y_test, proba_test, thr=thr_sel)
print("\n===== TEST 2024 con Umbral elegido =====")
print("Confusion matrix:")
print(np.array([[ev['tn'], ev['fp']], [ev['fn'], ev['tp']]]))
print("\nClassification report:")
print(classification_report(y_test, (proba_test >= thr_sel).astype(int)))
print(f"\nPredichos positivos: {100*( (proba_test >= thr_sel).mean() ): .3f}%  |  TP={ev['tp']}  FP={ev['fp']}  FN={ev['fn']}  TN={ev['tn']}")

# ==========================
# Guardado del modelo
# ==========================
artifact = {
    "model": best_est,
    "features": FEATURES,
    "meta": {
        "created_at": pd.Timestamp.now().isoformat(),
        "best_combo_eval_base": best,   # mejor combo en la comparación base (umbral 0.5)
        "threshold_selected": float(thr_sel),
        "precision_target": float(MIN_PRECISION),
        "val_metrics_at_thr": {"precision": float(p_val), "recall": float(r_val)},
        "test_metrics_at_thr": {
            "precision": float(ev["precision"]),
            "recall": float(ev["recall"]),
            "f1": float(ev["f1"]),
            "pr_auc": float(average_precision_score(y_test, proba_test)),
            "roc_auc": float(roc_auc_score(y_test, proba_test))
        }
    }
}
fname = f"best_model_OPT_{best_name}_{best_bal}.joblib".replace(" ","_")
save_path = os.path.join(SAVE_DIR, fname)
joblib.dump(artifact, save_path)
print(f"\n💾 Modelo optimizado guardado en: {save_path}")

# CSV comparativo
csv_path = os.path.join(SAVE_DIR, "model_comparison_pruned.csv")
df_res.to_csv(csv_path, index=False)
print(f"💾 Resultados comparativos guardados en: {csv_path}")

headline("✅ Pipeline finalizado")



🚀 Iniciando pipeline de predicción de accidentes
------------------------------------------------
Dataset cargado: 3,495,160 registros, 16 columnas

Distribución de clases:
Clase 0 (No accidente): 3,427,736 (98.07%)
Clase 1 (Accidente): 67,424 (1.93%)

📊 División temporal:
Entrenamiento: 2,604,597 registros (74.5%)
Validación (dic-2023): 37,938 registros
Test (2024): 447,612 registros
Accidentes en train: 52,167 (2.00%)
Accidentes en val  : 586 (1.54%)
Accidentes en test : 7,536 (1.68%)

🧪 Feature Engineering por cluster (evitando leakage)
----------------------------------------------------

⚖️ Creando datasets balanceados (ligeros y estables)
----------------------------------------------------
   ✅ USC 1:20  → 1,095,507 filas | pos=52,167 (4.76%)
   ✅ USM 1:10  → 573,837 filas | pos=52,167 (9.09%)
   Modelos: ['RandomForest_HighRecall', 'LightGBM_Balanced', 'XGBoost_Weighted']

🔬 Evaluando combinaciones…
--------------------------

  📊 Técnica: UNDERSAMPLING_CONSERVATIVE  | datos=1